# Figure 12.6 — Interactive SVM Margin Geometry
### *Hands-On Electroretinography in the Age of AI* (Apress)
**Chapter 12: Classical Machine Learning for ERG Classification**  
Author: Hamid Tavakoli | London, 2026

---

This notebook reproduces **Figure 12.6** from the manuscript as a fully interactive simulation.
It is designed for readers who wish to build an intuitive understanding of how the Support Vector
Machine (SVM) works — no prior knowledge of machine learning is required.

**What this figure shows**  
Each point represents one ERG recording plotted in a two-dimensional feature space using the two
highest-importance features identified in §12.2.4 of the chapter:

- **x-axis** — b-wave implicit time (normalised): short in normal retinas, delayed in pathology
- **y-axis** — Spectrogram mean intensity (b_mean): high in normal retinas, reduced in pathology

The SVM finds the single straight boundary that separates the two classes with the **largest
possible gap** — the margin. Only the recordings nearest that boundary — the **support vectors** —
determine where it goes. All other recordings are irrelevant to the boundary once training is
complete.

---
> **Note for Colab users:** Run the cells in order. If widgets do not appear after the last cell,
> go to *Runtime → Restart and run all*.


In [170]:
# ── Install / enable ipywidgets (Colab only; safe to run in standard Jupyter) ──
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Colab widget manager enabled.")
except ImportError:
    pass  # Standard Jupyter — no action needed

try:
    import ipywidgets
    print(f"ipywidgets {ipywidgets.__version__} ready.")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ipywidgets", "-q"])
    print("ipywidgets installed.")


Colab widget manager enabled.
ipywidgets 7.7.1 ready.


In [171]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.family"] = "DejaVu Sans"


## Synthetic ERG feature data

The two point clouds are **synthetic but clinically grounded**:

| Class | Implicit time | b_mean | Clinical meaning |
|---|---|---|---|
| Normal (blue) | Short (low x) | High (high y) | Intact bipolar cell function |
| Abnormal (red) | Delayed (high x) | Reduced (low y) | Bipolar cell dysfunction |

A fixed random seed (`42`) ensures the figure is fully reproducible.


In [172]:
# ── Reproducible synthetic ERG feature data ────────────────────────────
rng = np.random.default_rng(42)

# Normal ERGs: short b-wave implicit time (low x), high b_mean (high y)
n_normal  = 20
normal_x  = rng.normal(loc=0.18, scale=0.07, size=n_normal).clip(0.03, 0.40)
normal_y  = rng.normal(loc=0.80, scale=0.08, size=n_normal).clip(0.55, 0.98)

# Abnormal ERGs: delayed implicit time (high x), reduced b_mean (low y)
n_abnormal  = 20
abnormal_x  = rng.normal(loc=0.72, scale=0.08, size=n_abnormal).clip(0.48, 0.95)
abnormal_y  = rng.normal(loc=0.25, scale=0.08, size=n_abnormal).clip(0.05, 0.52)

# Support vector indices — points closest to the diagonal boundary
sv_normal_idx   = [4, 8, 13]
sv_abnormal_idx = [2, 7, 11]

# Colour palette
COL_NORMAL_FACE   = "#B5D4F4"
COL_NORMAL_EDGE   = "#185FA5"
COL_ABNORMAL_FACE = "#F7C1C1"
COL_ABNORMAL_EDGE = "#A32D2D"
COL_BOUNDARY      = "#185FA5"
COL_MARGIN        = "#555555"
COL_AXIS_BG       = "#F5F4EF"

print(f"Normal ERGs    | mean implicit time = {normal_x.mean():.2f} | mean b_mean = {normal_y.mean():.2f}")
print(f"Abnormal ERGs  | mean implicit time = {abnormal_x.mean():.2f} | mean b_mean = {abnormal_y.mean():.2f}")


Normal ERGs    | mean implicit time = 0.18 | mean b_mean = 0.81
Abnormal ERGs  | mean implicit time = 0.73 | mean b_mean = 0.24


## Drawing function

`draw_svm()` renders the complete figure for any combination of widget parameters.
Read the inline comments to understand what each section draws and why.


In [173]:
def draw_svm(margin_width, angle_offset, show_sv,
             show_zones, show_margin_arrow, show_pathophys):

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_facecolor(COL_AXIS_BG)
    fig.patch.set_facecolor("white")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(
        "b-wave implicit time (normalised)  →\n"
        "[ short = normal   |   delayed = pathological ]",
        fontsize=11, labelpad=10)
    ax.set_ylabel(
        "Spectrogram mean intensity — b_mean  →\n"
        "[ reduced = pathological   |   high = normal ]",
        fontsize=11, labelpad=10)
    ax.set_title(
        "Figure 12.6 — SVM Maximum-Margin Decision Boundary\n"
        "ERG classification: Normal vs Abnormal recordings",
        fontsize=13, fontweight="bold", pad=14)
    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_edgecolor("#CCCCCC")

    # ── Decision boundary geometry ──────────────────────────────────────
    # Base angle -45° runs top-left to bottom-right, separating the clouds.
    # User adjusts ±30° around this base via the angle slider.
    theta  = np.deg2rad(-45 + angle_offset)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    # Boundary line through centre (0.5, 0.5)
    t_vals = np.linspace(-1.5, 1.5, 300)
    bx = 0.5 + t_vals * cos_t
    by = 0.5 + t_vals * sin_t

    # Perpendicular unit vector (points toward normal zone = top-left)
    perp_x = -sin_t
    perp_y =  cos_t

    # Margin half-width in data units (slider 10–95 → ~0.05–0.475)
    m = margin_width / 200.0

    # Margin boundary lines (dashed)
    mx1, my1 = bx + perp_x * m, by + perp_y * m   # upper margin
    mx2, my2 = bx - perp_x * m, by - perp_y * m   # lower margin

    # ── Shaded class zones ──────────────────────────────────────────────
    if show_zones:
        from matplotlib.patches import Polygon
        from matplotlib.collections import PatchCollection
        corners_all = [[-5,-5],[5,-5],[5,5],[-5,5]]

        # Normal zone: all points on the perp_x/perp_y side of margin-top line
        # A point P is in the normal zone if dot(P - point_on_mx1, perp) > 0
        ref = np.array([mx1[150], my1[150]])
        normal_corners = [c for c in corners_all
                          if (np.array(c) - ref).dot([perp_x, perp_y]) > 0]
        margin_top_pts = [[mx1[0], my1[0]], [mx1[-1], my1[-1]]]
        nz_poly = Polygon(margin_top_pts + normal_corners,
                          closed=True, facecolor=COL_NORMAL_FACE,
                          alpha=0.20, zorder=0)
        ax.add_patch(nz_poly)

        # Abnormal zone: opposite side of margin-bottom line
        ref2 = np.array([mx2[150], my2[150]])
        abnormal_corners = [c for c in corners_all
                            if (np.array(c) - ref2).dot([perp_x, perp_y]) < 0]
        margin_bot_pts = [[mx2[0], my2[0]], [mx2[-1], my2[-1]]]
        az_poly = Polygon(margin_bot_pts + abnormal_corners,
                          closed=True, facecolor=COL_ABNORMAL_FACE,
                          alpha=0.20, zorder=0)
        ax.add_patch(az_poly)

    ax.plot(mx1, my1, "--", color=COL_MARGIN, linewidth=1.0, alpha=0.75, zorder=2)
    ax.plot(mx2, my2, "--", color=COL_MARGIN, linewidth=1.0, alpha=0.75, zorder=2)
    ax.plot(bx,  by,  "-",  color=COL_BOUNDARY, linewidth=2.5, zorder=3)

    # ── Margin width annotation ─────────────────────────────────────────
    if show_margin_arrow:
        t_ann  = 0.55
        ax_ann = 0.5 + t_ann * cos_t
        ay_ann = 0.5 + t_ann * sin_t
        p1 = np.array([ax_ann + perp_x * m, ay_ann + perp_y * m])
        p2 = np.array([ax_ann - perp_x * m, ay_ann - perp_y * m])
        ax.annotate("", xy=p2, xytext=p1,
                    arrowprops=dict(arrowstyle="<->", color="#555555",
                                   lw=1.2, shrinkA=0, shrinkB=0))
        mid = (p1 + p2) / 2
        ax.text(mid[0] + 0.03, mid[1], "margin", fontsize=9,
                color="#555555", va="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          ec="none", alpha=0.85))

    # ── Data points ─────────────────────────────────────────────────────
    ax.scatter(normal_x, normal_y,
               s=70, facecolors=COL_NORMAL_FACE,
               edgecolors=COL_NORMAL_EDGE, linewidths=0.8, zorder=4)
    ax.scatter(abnormal_x, abnormal_y,
               s=70, facecolors=COL_ABNORMAL_FACE,
               edgecolors=COL_ABNORMAL_EDGE, linewidths=0.8, zorder=4)

    # ── Support vectors ─────────────────────────────────────────────────
    if show_sv:
        for i in sv_normal_idx:
            ax.scatter(normal_x[i], normal_y[i],
                       s=220, facecolors="none",
                       edgecolors=COL_NORMAL_EDGE,
                       linewidths=1.8, linestyles="--", zorder=5)
        for i in sv_abnormal_idx:
            ax.scatter(abnormal_x[i], abnormal_y[i],
                       s=220, facecolors="none",
                       edgecolors=COL_ABNORMAL_EDGE,
                       linewidths=1.8, linestyles="--", zorder=5)
        # Callout annotation
        sx, sy = normal_x[sv_normal_idx[0]], normal_y[sv_normal_idx[0]]
        ax.annotate("support vector\n(hardest to classify)",
                    xy=(sx, sy), xytext=(sx + 0.13, sy - 0.13),
                    fontsize=9, color="#333333",
                    arrowprops=dict(arrowstyle="->", color="#555555",
                                   connectionstyle="arc3,rad=-0.2"),
                    bbox=dict(boxstyle="round,pad=0.3", fc="white",
                              ec="#CCCCCC", alpha=0.92))

    # ── Region labels ───────────────────────────────────────────────────
    ax.text(0.35, 0.60, "← normal ERG region",
            fontsize=10, color=COL_NORMAL_EDGE, fontweight="medium", alpha=0.85)
    ax.text(0.55, 0.38, "abnormal ERG region →",
            fontsize=10, color=COL_ABNORMAL_EDGE, fontweight="medium", alpha=0.85)

    # ── Pathophysiology annotation boxes ────────────────────────────────
    if show_pathophys:
        ax.text(0.3, 0.97,
                "Normal ERG\n• Short implicit time\n• High b_mean\n• Intact bipolar cells",
                fontsize=8.5, color=COL_NORMAL_EDGE, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.4", fc="#E6F1FB",
                          ec=COL_NORMAL_EDGE, alpha=0.88, linewidth=0.8))
        ax.text(0.80, 0.03,
                "Abnormal ERG\n• Delayed implicit time\n• Reduced b_mean\n• Bipolar cell dysfunction",
                fontsize=8.5, color=COL_ABNORMAL_EDGE, va="bottom", ha="left",
                bbox=dict(boxstyle="round,pad=0.4", fc="#FCEBEB",
                          ec=COL_ABNORMAL_EDGE, alpha=0.88, linewidth=0.8))

    # ── Legend ──────────────────────────────────────────────────────────
    legend_elements = [
        mpatches.Patch(facecolor=COL_NORMAL_FACE, edgecolor=COL_NORMAL_EDGE,
                       linewidth=0.8, label="Normal ERG"),
        mpatches.Patch(facecolor=COL_ABNORMAL_FACE, edgecolor=COL_ABNORMAL_EDGE,
                       linewidth=0.8, label="Abnormal ERG"),
        Line2D([0],[0], color=COL_BOUNDARY, linewidth=2.5,
               label="Decision boundary"),
        Line2D([0],[0], color=COL_MARGIN, linewidth=1.0,
               linestyle="--", label="Margin edges"),
    ]
    if show_sv:
        legend_elements.append(
            Line2D([0],[0], marker="o", color="w", markerfacecolor="none",
                   markeredgecolor="#555555", markeredgewidth=1.5,
                   markersize=10, linestyle="--", label="Support vectors"))

    ax.legend(handles=legend_elements, loc="upper right",
              fontsize=9, framealpha=0.92, edgecolor="#CCCCCC")

    plt.tight_layout()
    plt.show()


## Interactive figure — run this cell to launch

Use the controls to explore the three core SVM concepts:

| Control | Concept it teaches |
|---|---|
| **Margin width** | The SVM maximises this gap — wider margin = tighter constraint on the boundary |
| **Boundary angle** | Rotating the line shows that only support vectors fix its position |
| **Highlight support vectors** | The decisive borderline minority become visible |
| **Pathophysiology notes** | Connects the feature axes to real ERG biology (§12.1.1) |


In [174]:
style  = {"description_width": "200px"}
layout = widgets.Layout(width="350px")

w_margin = widgets.IntSlider(
    value=52, min=10, max=95, step=1,
    description="Margin width",
    style=style, layout=layout, continuous_update=True)

w_angle = widgets.IntSlider(
    value=0, min=-120, max=120, step=1,
    description="Boundary angle offset (°)",
    style=style, layout=layout, continuous_update=True)

w_sv = widgets.Checkbox(
    value=True, description="Highlight support vectors",
    style=style, layout=layout)

w_zones = widgets.Checkbox(
    value=True, description="Show class zones (shading)",
    style=style, layout=layout)

w_arrow = widgets.Checkbox(
    value=True, description="Show margin annotation",
    style=style, layout=layout)

w_pathophys = widgets.Checkbox(
    value=True, description="Show pathophysiology notes",
    style=style, layout=layout)

ui = widgets.VBox([
    widgets.HTML(
        "<b style='font-size:14px'>Figure 12.6 — SVM Interactive Controls</b><br>"
        "<span style='font-size:11px;color:#666'>"
        "Adjust the controls and observe how only the circled support vectors "
        "determine the boundary position, regardless of where all other points lie."
        "</span>"
    ),
    w_margin, w_angle,
    widgets.HBox([w_sv, w_zones]),
    widgets.HBox([w_arrow, w_pathophys]),
])

out = widgets.interactive_output(
    draw_svm,
    dict(margin_width=w_margin,
         angle_offset=w_angle,
         show_sv=w_sv,
         show_zones=w_zones,
         show_margin_arrow=w_arrow,
         show_pathophys=w_pathophys))

display(ui, out)


Output()

## What to explore with each control

**Margin width slider**  
Widen the margin and notice how the boundary is forced to sit precisely between the two clouds.
Narrow it and the boundary can drift almost anywhere. The SVM always selects the *widest* margin
that still separates both classes correctly. This is its defining geometric property.

**Boundary angle slider**  
Rotate the boundary. The non-circled points have no effect on where it settles. Only the support
vectors — the circled recordings nearest the margin edges — constrain the boundary. In clinical
terms, these are the patients whose retinal function sits closest to the threshold between normal
and pathological.

**Highlight support vectors**  
Toggle off: all forty recordings look equally important.  
Toggle on: the decisive minority become visible. In a dataset of thousands of ERG recordings, the
final fitted SVM boundary may be determined by as few as five or ten support vectors.

**Pathophysiology notes**  
The axis directions encode real ERG biology. Delayed b-wave implicit time and reduced spectrogram
mean intensity (b_mean) are the two features with the highest Random Forest importance score in
§12.2.4, and both reflect slowed or attenuated signal transmission through the bipolar cell layer
of the outer retina.

---

## References

Albasu, F. et al. (2024) "Electroretinogram analysis using a short-time Fourier transform and
machine learning techniques", *Bioengineering*, 11(9), p. 866.
doi:[10.3390/bioengineering11090866](https://doi.org/10.3390/bioengineering11090866)

Cortes, C. and Vapnik, V. (1995) "Support-vector networks", *Machine Learning*, 20(3), pp. 273–297.
doi:[10.1007/BF00994018](https://doi.org/10.1007/BF00994018)

Platt, J. (1999) "Probabilistic outputs for support vector machines", in *Advances in Large Margin
Classifiers*. MIT Press, pp. 61–74.
